# Capítulo 13 — Sistemas de Recomendação

**Livro:** começa na página **247** do PDF.

**Gabarito** dos exercícios do caderno de prática. Tente sozinho antes de olhar.

## Preparação

In [ ]:
set.seed(42)
options(digits = 4, repr.plot.width = 7, repr.plot.height = 4.5)
suppressPackageStartupMessages({ library(glmnet); library(rpart); library(randomForest); library(FNN); library(e1071) })
cat("Pronto.", R.version.string, "
")

## Filtro Colaborativo Baseado no Usuário · PDF p. 248

#### Exercício: Recomendação por usuários parecidos

Monte uma matriz de notas 5 usuários × 4 filmes com alguns `NA`. Calcule a similaridade de cosseno entre usuários (usando só os filmes avaliados por ambos) e preveja a nota que falta do usuário 1 como média ponderada pelas similaridades.

In [ ]:
notas <- matrix(c(5, 4, NA, 1,
                  4, 5, 2, 1,
                  1, 2, 5, 4,
                  5, 5, 1, NA,
                  2, 1, 4, 5), 5, byrow = TRUE, dimnames = list(paste0("u", 1:5), c("A", "B", "C", "D")))
cosseno <- function(a, b) { ok <- !is.na(a) & !is.na(b); sum(a[ok] * b[ok]) / sqrt(sum(a[ok]^2) * sum(b[ok]^2)) }
sim <- sapply(2:5, function(j) cosseno(notas[1, ], notas[j, ]))
avaliaram <- !is.na(notas[2:5, "C"])
sum(sim[avaliaram] * notas[2:5, "C"][avaliaram]) / sum(sim[avaliaram])

## FunkSVD · PDF p. 249

#### Exercício: Fatoração de matrizes (FunkSVD)

Na mesma matriz, aprenda P (5×2) e Q (4×2) por gradiente descendente só nas notas observadas, minimizando (nota − pᵤ·qᵢ)² + 0,02(|pᵤ|² + |qᵢ|²). Mostre a matriz P·Qᵀ completa.

In [ ]:
notas <- matrix(c(5, 4, NA, 1, 4, 5, 2, 1, 1, 2, 5, 4, 5, 5, 1, NA, 2, 1, 4, 5), 5, byrow = TRUE)
P <- matrix(runif(10, 0, 1), 5); Q <- matrix(runif(8, 0, 1), 4)
obs <- which(!is.na(notas), arr.ind = TRUE)
for (epoca in 1:3000) for (r in seq_len(nrow(obs))) {
  u <- obs[r, 1]; i <- obs[r, 2]
  erro <- notas[u, i] - sum(P[u, ] * Q[i, ])
  pu <- P[u, ]
  P[u, ] <- P[u, ] + 0.01 * (erro * Q[i, ] - 0.02 * P[u, ])
  Q[i, ] <- Q[i, ] + 0.01 * (erro * pu - 0.02 * Q[i, ])
}
round(P %*% t(Q), 1)